# V2 - Phase 4 - Entrainement par etiquette

**Objectif.** Entrainer cinq modeles HGB V2 *distincts*, un par etiquette :

- `continuity_risk_12m_label` (cible composite, reference de comparaison avec V1)
- `legal_distress_risk_12m_label` (procedures collectives)
- `radiation_risk_12m_label` (cessations volontaires + clotures INSEE)
- `financial_weakness_risk_12m_label` (faiblesse financiere ex-ante)
- `filing_anomaly_risk_12m_label` (anomalie de depot des comptes)

**Pourquoi.** En V1, on agrege quatre evenements aux dynamiques predictives tres differentes en un seul label composite. Les procedures collectives (juge) sont mieux predites que les cessations volontaires (intention humaine). Un modele par label expose des probabilites *interpretables* a l'utilisateur final et permet de cibler le 'risque de procedure collective' separement du 'risque de radiation'.

**Variations vs V1.**
- HGB uniquement (V1 comparait 4 librairies; on a choisi HGB d'apres V1 Run 8).
- **Pas de `class_weight='balanced'`**. V2 laisse les probabilites brutes ; le seuil de decision sera tune en Phase 5 et la calibration validee en Phase 8. Sans `class_weight`, ECE attendu plus bas.
- Hyperparametres : ceux trouves en V1 Phase B (`docs/ouputs/ml-artifacts/tuned_params_hgb.json`) sauf si la cible specifique demande mieux (audit ulterieur).

**Critere de validation pour passer en Phase 5.** AP de `legal_distress_risk_12m_label` >= 0.30 sur le test 2023 (procedures collectives mieux predites que la cible composite V1 a AP 0.30). Si oui, la decomposition par label est validee.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-v2'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE_DRIVE = f'{DRIVE_ROOT}/data-lake'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

DATA_LAKE_LOCAL = '/content/pfein_phase4_lake'
ARTIFACTS_LOCAL = '/content/pfein_phase4_artifacts'
ARTIFACTS_DRIVE = f'{DRIVE_ROOT}/ml-artifacts'

TUNED_PARAMS_PATH = f'{BACKEND_DIR}/docs/ouputs/ml-artifacts/tuned_params_hgb.json'

LABELS = [
    'continuity_risk_12m_label',
    'legal_distress_risk_12m_label',
    'radiation_risk_12m_label',
    'financial_weakness_risk_12m_label',
    'filing_anomaly_risk_12m_label',
]

# Train sample size. V1's tuned hyperparams were fit at 2M rows, so we
# stay in that regime so the params transfer cleanly. Set to None for the
# full ~191M-row V2 (will take ~30-60 min per label).
MAX_ROWS_PER_TRAIN = 2_000_000

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)
Path(DATA_LAKE_LOCAL).mkdir(parents=True, exist_ok=True)
Path(ARTIFACTS_LOCAL).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR        =', BACKEND_DIR)
print('DATA_LAKE_LOCAL    =', DATA_LAKE_LOCAL)
print('ARTIFACTS_LOCAL    =', ARTIFACTS_LOCAL)
print('TUNED_PARAMS_PATH  =', TUNED_PARAMS_PATH)
print('MAX_ROWS_PER_TRAIN =', MAX_ROWS_PER_TRAIN)
print('LABELS             =', LABELS)

In [ ]:
import os, subprocess

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

assert Path(TUNED_PARAMS_PATH).exists(), f'Tuned params not found at {TUNED_PARAMS_PATH}'
print('\nTuned params (V1 Phase B):')
!cat "$TUNED_PARAMS_PATH"

## 2. Symlink les donnees V2 dans le data-lake local

Le trainer V2 lit `features/company_year_features_v2/` et `features/risk_labels_v2/`. On les pointe via symlinks depuis Drive (lectures seulement, pas d'ecriture sur Drive en plein training).

In [ ]:
import os

features_local = Path(DATA_LAKE_LOCAL) / 'features'
features_local.mkdir(parents=True, exist_ok=True)

shared_features = ['company_year_features_v2', 'risk_labels_v2']
for t in shared_features:
    src = Path(DATA_LAKE_DRIVE) / 'features' / t
    dst = features_local / t
    if dst.is_symlink() or dst.exists():
        if dst.is_symlink():
            dst.unlink()
        else:
            import shutil as _sh; _sh.rmtree(dst)
    if not src.exists():
        print(f'ERR Missing source on Drive: {src}')
        continue
    os.symlink(src, dst)
    print(f'symlinked {dst} -> {src}')

print('\nLocal data-lake tree:')
!ls -la "$DATA_LAKE_LOCAL/features"

## 3. Boucle d'entrainement -- un fit par etiquette

Chaque appel lance `app.tools.v2.train_label_specific_model` en sous-processus :
- lit features V2 + labels V2,
- echantillonne `MAX_ROWS_PER_TRAIN` lignes (hash-deterministe sur siren | prediction_year),
- split temporel : train < 2023, test = 2023,
- fit HGB avec les hyperparams tunes V1 Phase B (sans `class_weight`),
- ecrit `ml-artifacts/v2/per_label/<label>/{model.joblib, metadata.json, run_summary.md, test_predictions.parquet}`.

In [ ]:
import shlex, subprocess, sys, time, json

results = {}
for label in LABELS:
    cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.v2.train_label_specific_model',
        '--data-lake-dir', DATA_LAKE_LOCAL,
        '--artifacts-dir', ARTIFACTS_LOCAL,
        '--target', label,
        '--train-start-year', '2017',
        '--train-end-year',   '2023',
        '--params-file',      TUNED_PARAMS_PATH,
    ]
    if MAX_ROWS_PER_TRAIN is not None:
        cmd += ['--max-rows', str(MAX_ROWS_PER_TRAIN)]

    print('=' * 80)
    print('Target :', label)
    print(' '.join(shlex.quote(p) for p in cmd))
    print('-' * 80)
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print('STDERR (tail):')
        print(proc.stderr[-3000:])
        raise SystemExit(f'Training failed for {label} with code {proc.returncode}')
    print(f'Duree : {elapsed/60:.1f} min')

    meta_path = Path(ARTIFACTS_LOCAL) / 'v2' / 'per_label' / label / 'metadata.json'
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    results[label] = meta
    print(f'  AP={meta["metrics"]["average_precision"]:.4f}  AUC={meta["metrics"]["roc_auc"]:.4f}  '
          f'test_pos_rate={meta["metrics"]["test_positive_rate"]:.4f}')

print('\nAll 5 trainings done.')

## 4. Tableau comparatif inter-etiquettes (test 2023)

In [ ]:
import pandas as pd

rows = []
for label, meta in results.items():
    m = meta['metrics']
    rows.append({
        'label'            : label.replace('_12m_label', ''),
        'train_rows'       : meta['train_rows'],
        'test_rows'        : meta['test_rows'],
        'test_positives'   : meta['test_positives'],
        'test_pos_rate'    : m['test_positive_rate'],
        'AP'               : m['average_precision'],
        'AUC'              : m['roc_auc'],
        'F1@0.5'           : m['f1_at_0_5'],
        'precision@0.5'    : m['precision_at_0_5'],
        'recall@0.5'       : m['recall_at_0_5'],
    })
summary = pd.DataFrame(rows).set_index('label')
print(summary.round(4).to_string())

legal_distress_ap = summary.loc['legal_distress_risk', 'AP']
print()
if legal_distress_ap >= 0.30:
    print(f'[OK] legal_distress AP = {legal_distress_ap:.3f} >= 0.30 (gate Phase 4 atteint). Passer en Phase 5 (tuning seuils).')
else:
    print(f'[KO] legal_distress AP = {legal_distress_ap:.3f} < 0.30 (gate non atteint). Investiguer avant Phase 5 :')
    print('     - distribution des labels par annee (verifier pas de regression dans risk_labels_v2)')
    print('     - feature importance HGB (verifier que les features d\'identite ont du signal sur ce label specifique)')
    print('     - relancer sans --max-rows pour valider que ce n\'est pas un effet d\'echantillonnage')

## 5. Sync ml-artifacts/v2/per_label/ vers Drive

In [ ]:
import shutil, json as _json

src_root = Path(ARTIFACTS_LOCAL) / 'v2' / 'per_label'
dst_root = Path(ARTIFACTS_DRIVE) / 'v2' / 'per_label'
dst_root.mkdir(parents=True, exist_ok=True)

for label in LABELS:
    src = src_root / label
    dst = dst_root / label
    if not src.exists():
        print(f'WARN: {src} missing')
        continue
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Copied {src} -> {dst}')

print('\nDrive contents:')
!ls -la "$ARTIFACTS_DRIVE/v2/per_label"

## 6. Reporter dans `docs/v2/v2_phase_log.md`

Copier le tableau ci-dessous dans la section Phase 4 du journal.

In [ ]:
from datetime import datetime

log_lines = [
    f'**Date d\'execution :** {datetime.now().strftime("%Y-%m-%d")}',
    f'**Notebook :** `collabs/v2/v2_phase_4_train_per_label.ipynb`',
    f'**Script :** `app/tools/v2/train_label_specific_model.py`',
    f'**Echantillon par fit :** {MAX_ROWS_PER_TRAIN:,} rows' if MAX_ROWS_PER_TRAIN else '**Echantillon :** full V2 features',
    '',
    '### Resultats -- metriques par etiquette (test 2023)',
    '',
    '| Etiquette | Taux positifs | AP | AUC | F1@0.5 | Precision@0.5 | Recall@0.5 |',
    '|---|---:|---:|---:|---:|---:|---:|',
]
for label, meta in results.items():
    m = meta['metrics']
    log_lines.append(
        f'| `{label}` | {m["test_positive_rate"]:.4%} | {m["average_precision"]:.4f} | '
        f'{m["roc_auc"]:.4f} | {m["f1_at_0_5"]:.4f} | {m["precision_at_0_5"]:.4f} | '
        f'{m["recall_at_0_5"]:.4f} |'
    )
print('\n'.join(log_lines))